In [8]:
%%capture

from crewai import Agent, Task, Crew, Process
from crewai import LLM
from crewai_tools import PDFSearchTool, SerperDevTool

In [9]:
import litellm
litellm.ssl_verify = False

---


In [10]:
llm = LLM(
    model="watsonx/ibm/granite-4-h-small",
    base_url="https://us-south.ml.cloud.ibm.com",
    project_id="skills-network",
)

### Initializing Serper Dev Tool: 


In [11]:
import os
os.environ['SERPER_API_KEY'] = '0362f4ac0824d3a9f5e33f47edd95170c3b93b45' 

In [12]:
web_search_tool = SerperDevTool()

### Creating our PDF Search Tool: 


In [14]:
import warnings

def warn( *args, **kwargs):
    pass
warnings.filterwarnings('ignore') #Keeps Jupyter Notebook clean (not part of functionality)
warnings.warn = warn


pdf_search_tool = PDFSearchTool(
    pdf="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf",
    config=dict(
        embedder=dict(
            provider="huggingface",
            config=dict(
                model="sentence-transformers/all-MiniLM-L6-v2"
            )
        )
    )
)

In [15]:
agent_centric_agent = Agent(
    role="The Daily Dish Inquiry Specialist",
    goal="""Accurately answer customer questions about The Daily Dish restaurant. 
    You must decide whether to use the restaurant's FAQ PDF or a web search to find the best answer.""",
    backstory="""You are an AI assistant for 'The Daily Dish'.
    You have access to two tools: one for searching the restaurant's FAQ document and another for searching the web.
    Your job is to analyze the user's question and choose the most appropriate tool to find the information needed to provide a helpful response.""",
    tools=[pdf_search_tool, web_search_tool],
    verbose=True,
    allow_delegation=False,
    llm=llm
)

#### **Step 1.2: Define the Task**

We create a single, broad task that instructs the agent to handle the customer's query.


In [16]:
agent_centric_task = Task(
    description="Answer the following customer query: '{customer_query}'. "
                "Analyze the question and use the tools at your disposal (PDF search or web search) to find the most relevant information. "
                "Synthesize the findings into a clear and friendly response.",
    expected_output="A comprehensive and well-formatted answer to the customer's query.",
    agent=agent_centric_agent
)

#### **Step 1.3: Assemble the Crew**

Finally, we create the Crew. It's a simple setup with our one agent and one task.


In [17]:
agent_centric_crew = Crew(
    agents=[agent_centric_agent],
    tasks=[agent_centric_task],
    process=Process.sequential,
    verbose=False
)

Before executing the crew, let's download the FAQ file that we have curated with all the questions and answers. As you execute the below line of code, the file will be downloaded, which you will be able to access from the file library on the left of your screen.


In [18]:
# Download the FAQ document for the tool to use
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf

--2026-08-03 20:15:40--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
200 OKequest sent, awaiting response... 
Length: 53993 (53K) [application/pdf]
Saving to: ‘The-Daily-Dish-FAQ.pdf’

The-Daily-Dish-FAQ. 100%[===================>]  52.73K  --.-KB/s    in 0.002s  

2026-08-03 20:15:40 (26.5 MB/s) - ‘The-Daily-Dish-FAQ.pdf’ saved [53993/53993]



Try asking the following questions:

1. What are the timings?
2. What is the phone number?
3. What is the location?

You could also ask some combinations of other questions that you will find in the FAQ PDF itself.


In [20]:
print("\nWelcome to The Daily Dish Chatbot!")
print("What would you like to know? (Type 'exit' to quit)")

while True: 
    user_input = input("\nYour question: ").lower()
    if user_input == 'exit':
        print("Thank you for chatting. Have a great day!")
        break
    
    if not user_input:
        print("Please type a question.")
        continue

    try:
        # Here we use our more advanced, task-centric crew
        result_agent_centric = agent_centric_crew.kickoff(inputs={'customer_query': user_input})
        print("\n--- The Daily Dish Assistant ---")
        print(result_agent_centric)
        print("--------------------------------")
    except Exception as e:
        print(f"An error occurred: {e}")


Welcome to The Daily Dish Chatbot!
What would you like to know? (Type 'exit' to quit)



Your question:  exit


Thank you for chatting. Have a great day!


In [21]:
task_centric_agent = Agent(
    role="Customer Service Specialist",
    goal="Provide exceptional customer service by following a multi-step process to answer customer questions accurately.",
    backstory="""You are an AI assistant for 'The Daily Dish'.
    You are an expert at following instructions. You will be given a sequence of tasks to complete.
    For each task, you will be provided with the specific tool needed to accomplish it.
    Your job is to execute each task diligently and pass the results to the next step.""",
    tools=[], # The agent is not given any tools directly
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [22]:
faq_search_task = Task(
    description="Search the restaurant's FAQ PDF for information related to the customer's query: '{customer_query}'.",
    expected_output="A snippet of the most relevant information from the PDF, or a statement that the information was not found.",
    tools=[pdf_search_tool], # Tool assigned directly to the task
    agent=task_centric_agent
)

response_drafting_task = Task(
    description="Using the information gathered from the FAQ search, draft a friendly and comprehensive response to the customer's query: '{customer_query}'.",
    expected_output="The final, customer-facing response.",
    agent=task_centric_agent,
    context=[faq_search_task]
)

In [23]:
task_centric_crew = Crew(
    agents=[task_centric_agent],
    tasks=[faq_search_task, response_drafting_task],
    process=Process.sequential,
    verbose=True
)

In [24]:
print("\nWelcome to The Daily Dish Chatbot!")
print("What would you like to know? (Type 'exit' to quit)")

while True: 
    user_input = input("\nYour question: ").lower()
    if user_input == 'exit':
        print("Thank you for chatting. Have a great day!")
        break
    
    if not user_input:
        print("Please type a question.")
        continue

    try:
        # Here we use our more advanced, task-centric crew
        result_task_centric = task_centric_crew.kickoff(inputs={'customer_query': user_input})
        print("\n--- The Daily Dish Assistant ---")
        print(result_task_centric)
        print("--------------------------------")
    except Exception as e:
        print(f"An error occurred: {e}")


Welcome to The Daily Dish Chatbot!
What would you like to know? (Type 'exit' to quit)



Your question:  Top dish to be tried for sure, what is that currently


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 78205566-141f-4464-991b-9f127067d149                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Service Specialist                                                                             │
│                                                                                                                 │
│  Task: Search the restaurant's FAQ PDF for information related to the customer's query: 'top dish to be tried   │
│  for sure, what is that currently'.                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: b34b2ef8-8ea9-4e04-bdf7-3574b0608de4                                                                     │
│  Agent: Customer Service Specialist                                                                             │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 78205566-141f-4464-991b-9f127067d149                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: We love helping guests discover new favorite dishes! Based on what our dining guests have loved  │
│  again and again, I'd recommend trying either our Chef's Signature Salmon with Lemon-Dill Risotto or our Spicy  │
│  Chorizo & Manchego Flatbread. Both of these dishes continue to be perennial favorites and come highly praised  │
│  by our regulars. The salmon is perfectly cooked and so flavorful, served with a bright lemon-dill risotto      │
│  that makes the whole dish shine. And if you're in the mood for something with a bit more kick, the chorizo     │
│  flatbread hits the spot with its delicious spicy sausage, sharp manchego cheese, and crispy flatbread crust.   │
│  Either way you'll be treated to an amazing, standout meal. Let us know which one you choose, we think you'll   │
│  be very happy with either selection!                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- The Daily Dish Assistant ---
We love helping guests discover new favorite dishes! Based on what our dining guests have loved again and again, I'd recommend trying either our Chef's Signature Salmon with Lemon-Dill Risotto or our Spicy Chorizo & Manchego Flatbread. Both of these dishes continue to be perennial favorites and come highly praised by our regulars. The salmon is perfectly cooked and so flavorful, served with a bright lemon-dill risotto that makes the whole dish shine. And if you're in the mood for something with a bit more kick, the chorizo flatbread hits the spot with its delicious spicy sausage, sharp manchego cheese, and crispy flatbread crust. Either way you'll be treated to an amazing, standout meal. Let us know which one you choose, we think you'll be very happy with either selection!
--------------------------------



Your question:  exit


Thank you for chatting. Have a great day!


In [25]:
from crewai.tools import tool
import re

@tool("Add Two Numbers Tool")
def add_numbers(data: str) -> int:
    """
    Extracts and adds integers from the input string.
    Example input: 'add 1 and 2' or '[1,2,3,4]'
    Output: sum of the numbers
    """
    # Find all integers in the string
    numbers = list(map(int, re.findall(r'-?\d+', data)))
    return sum(numbers)

Now, let's create another tool called `multiply_numbers`. This tool takes a string input such as `"multiply 2 and 3"` or `"values are [2,3,4]"`, extracts all the integers from the text, and returns their **product** as an integer.


In [26]:
from functools import reduce

@tool("Multiply Numbers Tool")
def multiply_numbers(data: str) -> int:
    """
    Extracts and multiplies integers from the input string.
    Example input: 'multiply 2 and 3' or '[2,3,4]'
    Output: the product of all numbers found
    """
    numbers = list(map(int, re.findall(r'-?\d+', data)))
    return reduce(lambda x, y: x * y, numbers, 1)

Now, we create a `Calculator Agent` with a clear goal being to extract, add or multiply numbers based on a user's query. We provide the relevant backstory, the tools that we created, and the LLM itself.


In [27]:
calculator_agent = Agent(
    role="Calculator",
    goal="Extracts, adds, or multiplies numbers when asked, using the Add Two Numbers and Multiply Numbers tools.",
    backstory="An expert at parsing numeric instructions and computing sums or products.",
    tools=[add_numbers, multiply_numbers],
    llm=llm,
    allow_delegation=False
)

We also create a `Calculation Task` by providing a clear description, an expected output, and an agent.


In [28]:
calculation_task = Task(
    description="Extract numbers from '{numbers}' and either add or multiply them, depending on the natural-language instruction.",
    expected_output="An integer result (sum or product) based on the user’s request.",
    agent=calculator_agent
)

Now let's bring together the created agent and task in a `Crew`.


In [29]:
crew = Crew(
    agents=[calculator_agent],
    tasks=[calculation_task],
    # verbose=True #Uncomment this to see the steps taken to get the final answer
)

Let's run the crew by providing a user query in `.kickoff()` and check the output.


In [30]:
# Inputs for addition…
result = crew.kickoff(inputs={'numbers': 'please add 4, 5, and 6'})
print("Sum result:", result)

Sum result: 15


In [31]:
# Inputs for multiplication…
result = crew.kickoff(inputs={'numbers': 'multiply 7 and 8 also 9 dont forget 10'})
print("Product result:", result)

Product result: 5040
